# AI Call Moderator v4 — Real-Time 3-Step Assembly Line (asyncio.Queue)

Implements the architecture document's **3-step assembly line** for the ROCm Jupyter lab,
moderating 1–3 real Kaggle recordings with near-instant flagging. **Speed is the metric**:
every hop is timed, and the headline number is *transcript-ready → flag raised* in milliseconds.

```
STEP 1  THE EARS   audio in 5s chunks ──> Faster-Whisper (CPU int8) ──> transcript
                                  │  zero-copy, in-process
                                  ▼  asyncio.Queue  (one FIFO per call = order preserved)
STEP 2  THE BRAIN  worker awaits queue ──> regex pre-screen (0 tokens) ──> vLLM guided-JSON judge
                                  │  escalation rules (deterministic code)
                                  ▼  alert asyncio.Queue
STEP 3  THE ALARM  consumer flags INSTANTLY (flushed print) + sqlite :memory: audit DB
```

**Architecture-document mapping** (what was kept / changed and why):

| Document said | v4 does | Reason |
|---|---|---|
| Faster-Whisper, 5s chunks | ✅ same, CPU int8 | CTranslate2 has no ROCm backend; CPU int8 beats real time on 5s chunks and leaves 100% of the GPU to vLLM |
| in-memory `asyncio.Queue` handshakes | ✅ same — one queue **per call** | zero-copy, ~0 ms hop; per-call FIFO keeps turn order (escalation rules need it) while calls run in parallel |
| Qwen 2.5 32B Q6 via vLLM | Qwen3-4B-Instruct via vLLM | 8× fewer weights = lower latency & tokens; the judging task doesn't need 32B |
| PyQt6 GUI + pyqtSignal + OS alarm | flushed alert print + sqlite `:memory:` | no desktop GUI inside Jupyter; the handshake & latency measurement are identical |
| WebRTC/WebSocket ingress | chunked file streaming (optional real-time pacing) | same dataflow; swap the producer's source for a socket later — nothing downstream changes |
| (LangChain/LangGraph considered) | **not used** | a fixed 3-stage pipeline needs no graph framework; it would add overhead between transcript and flag |

## STEP 0 — Tab 1: spin up the server (one command)

Open a **Terminal tab** (`+` → Terminal) and run:

```bash
bash run_vllm_server.sh
```

It installs all pip requirements, repairs the starlette/fastapi conflict, and launches vLLM.
Leave it running; wait for `Application startup complete` (first run downloads ~8 GB).
Then come back here — **Tab 2** — and run the cells below one by one.

In [ ]:
# ============ CELL 1: clients + health check ============
import os, json, re, time, asyncio, sqlite3            # stdlib only: parsing, timing, concurrency, in-memory DB
from collections import defaultdict, deque             # defaultdict: metric buckets / deque: rolling context window
from openai import OpenAI, AsyncOpenAI                 # OpenAI-compatible clients for the LOCAL vLLM server

BASE_URL          = "http://localhost:8000/v1"         # where Tab 1's vLLM listens
SERVED_MODEL_NAME = "call-moderator-llm"               # must match --served-model-name in the runner
API_KEY           = "local-key-123"                    # must match --api-key in the runner

sync_client  = OpenAI(base_url=BASE_URL, api_key=API_KEY)        # used once, for the health check below
async_client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)   # used by the pipeline (async, non-blocking)

try:                                                   # health check: fail with INSTRUCTIONS, not a stack trace
    available_models = [m.id for m in sync_client.models.list().data]
except Exception:
    raise SystemExit("Cannot reach vLLM at " + BASE_URL + ".\n"
                     "Tab 1 first: bash run_vllm_server.sh  ->  wait for 'Application startup complete'.")
print("vLLM is serving:", available_models)            # expect ['call-moderator-llm']
assert SERVED_MODEL_NAME in available_models           # wrong --served-model-name would trip this

## CELL 2 — Policy, escalation rules & zero-token pre-screener

The rules the Brain enforces. Codes (`C1`, `R2`, …) keep prompts and outputs tiny — token
efficiency starts here. The regex pre-screener costs zero tokens and only produces *hints*
for the LLM to verify (a rep **refusing** an SSN must not be auto-flagged).

In [ ]:
# ============ CELL 2: policy + pre-screener ============
POLICY = {                                             # code -> (severity, definition)
    "C1": ("critical", "Improper sensitive data: asking for or reading out a full SSN, "
                       "full card number, CVV, or account password"),
    "C2": ("critical", "Threats, harassment, hate speech, or targeted abuse at a person"),
    "C3": ("critical", "Unethical conduct: soliciting tips/bribes, off-the-books deals, "
                       "falsifying records, or advising someone to lie"),
    "C4": ("high",     "Unauthorized commitment: promising refunds over $50 or outcomes "
                       "outside stated policy"),
    "R1": ("high",     "Rep makes/offers account changes WITHOUT identity verification"),
    "R3": ("high",     "Disclosing internal-only data or another customer's information"),
    "R2": ("medium",   "Rep rudeness: dismissive, sarcastic, blaming, or hostile tone"),
}
SEVERITY_BY_CODE = {code: severity for code, (severity, _) in POLICY.items()}   # quick lookup for the rules

COMPANY_POLICY_ALLOWANCES = (                          # what reps ARE allowed -> lets the judge tell refusal from violation
    "Refunds up to $50 may be issued instantly; larger amounts require supervisor approval. "
    "Identity verification = full name + last 4 of account + billing ZIP (NEVER full SSN, "
    "full card number, CVV, or password). Reps may offer one goodwill credit up to $25 per call.")

# Escalation = DETERMINISTIC CODE (auditable), not the model:
#   rule 1: any critical violation            -> flag immediately
#   rule 2: two high-severity violations      -> flag
#   rule 3: customer sentiment <= -2 twice in a row -> flag (supervisor assist)

PRESCREEN_PATTERNS = {                                 # obvious keyword tells, compiled per turn at ~microsecond cost
    "C1": [r"\b(full|entire|whole|complete)\b.{0,40}\b(ssn|social security|card number)",
           r"\bcvv\b", r"\bsecurity code\b.{0,20}\bback\b", r"\byour password\b"],
    "C2": [r"\b(idiot|stupid|moron|pathetic|shut up)\b",
           r"\bi('ll| will) (sue|hurt|ruin|find) you\b"],
    "C3": [r"\b(venmo|cash app|tip me|kickback)\b", r"\boff[- ]the[- ]books\b"],
    "C4": [r"(refund|credit|comp)\w*\b.{0,40}\$\s?\d{3,}",
           r"\$\s?\d{3,}.{0,40}\b(refund|credit)", r"\bi (promise|guarantee)\b"],
}

def keyword_prescreen(text: str) -> list:              # returns hint codes; 0 tokens, deterministic
    lowered = text.lower()                             # case-insensitive matching
    return [code for code, patterns in PRESCREEN_PATTERNS.items()
            if any(re.search(pattern, lowered) for pattern in patterns)]

print(f"{len(POLICY)} violation codes loaded; pre-screener ready")

## CELL 3 — Instrumented, bounded, schema-constrained generation

Every LLM call goes through `generate_json`: an **`asyncio.Semaphore`** bounds in-flight
requests (the client-side gate; vLLM's own scheduler queues/batches beyond it), **guided
JSON** makes off-schema output impossible, and every call records exact **tokens** (from
vLLM's `usage`) and **latency** per pipeline stage.

In [ ]:
# ============ CELL 3: generate_json (the only door to the LLM) ============
MAX_CONCURRENT_LLM_REQUESTS = 16                       # client-side cap; protects sockets at scale, GPU stays saturated
llm_request_slot = asyncio.Semaphore(MAX_CONCURRENT_LLM_REQUESTS)   # async gatekeeper shared by all workers

STAGE_TOKEN_USAGE = defaultdict(lambda: {"calls": 0, "prompt": 0, "completion": 0})  # tokens per pipeline stage
STAGE_LATENCY_MS  = defaultdict(list)                  # wall-clock ms per call, per stage (for avg/p95/max later)

async def generate_json(stage: str, system_prompt: str, user_prompt: str,
                        json_schema: dict, max_tokens: int = 64) -> dict:
    async with llm_request_slot:                       # wait here if 16 requests are already in flight
        started = time.perf_counter()                  # latency clock starts at dispatch
        response = await async_client.chat.completions.create(
            model=SERVED_MODEL_NAME,                   # the vLLM-served judge
            messages=[{"role": "system", "content": system_prompt},
                      {"role": "user",   "content": user_prompt}],
            temperature=0,                             # greedy decoding -> reproducible verdicts
            max_tokens=max_tokens,                     # hard output budget per call
            extra_body={"guided_json": json_schema},   # vLLM grammar-constrains sampling to this schema
        )
    STAGE_LATENCY_MS[stage].append((time.perf_counter() - started) * 1000)   # record ms
    bucket = STAGE_TOKEN_USAGE[stage]                  # accumulate exact server-reported token counts
    bucket["calls"]      += 1
    bucket["prompt"]     += response.usage.prompt_tokens
    bucket["completion"] += response.usage.completion_tokens
    try:
        return json.loads(response.choices[0].message.content)   # guided JSON -> this should never fail
    except json.JSONDecodeError:
        return {}                                      # belt-and-braces: empty verdict instead of a crash

# quick smoke test: proves server reachability, guided decoding, and metering in one shot
smoke = await generate_json("smoke_test", "Rate sentiment.",
                            'Customer said: "this is great, thanks!"',
                            {"type": "object", "properties": {"sentiment": {"enum": [-2, -1, 0, 1, 2]}},
                             "required": ["sentiment"]}, max_tokens=8)
print("guided JSON:", smoke, "| latency:", f"{STAGE_LATENCY_MS['smoke_test'][0]:.0f} ms")

## CELL 4 — Pull 1–3 recordings from Kaggle (MCP + OAuth 2.0)

Two cells: **4a connects & authenticates**, **4b discovers tools & downloads**. Both are built
around lessons learned against Kaggle's live MCP server:

**Authentication (4a):**

- Kaggle accepts **anonymous connections silently** — `list_tools()` works, but real calls fail
  with `An error occurred invoking '...'`. So 4a always *probes* identity with
  `get_user_profile` instead of assuming the connect implied auth.
- Kaggle's OAuth registration requires `token_endpoint_auth_method="none"` (public PKCE
  client). Without that line, registration dies with HTTP 400 before the consent URL appears.
- Kaggle's `authorize` tool replies **off-spec** (a bare string `"User is already authorized."`
  instead of an MCP content list) — the SDK's validator raises on it, so we catch and treat it
  as the success it usually is.
- Tokens are cached in a **kernel-global** (`KAGGLE_OAUTH_CACHE`), so re-running 4a reuses
  them instead of prompting for consent every time. Cleared only on kernel restart.
- `KAGGLE_TOKEN` (a `KGAT...` string from kaggle.com → Settings → API) bypasses the browser
  flow entirely via a Bearer header — useful when no browser is handy.

**Tool calls (4b) — schema-driven, never guessed:**

- Tool *names* are taken from the live `list_tools()` result (e.g. `search_datasets`,
  `download_dataset`), and tool *arguments* are derived from each tool's own **`inputSchema`**:
  `build_args()` reads the schema's required properties and places our value into the first
  required string field. The server's spec is the ground truth — both schemas are printed so
  any mismatch is visible instead of silent.
- Hand-written argument spellings remain as fallbacks after the schema-derived attempt, and
  `call_kaggle()` tries each variant in order, surfacing the last server error if all fail.
- **Search is discovery-only and non-fatal**: it helps you pick `DATASET_REFERENCE`, but if it
  errors the cell warns and proceeds to download the known dataset — a flaky optional step
  never blocks the pipeline.
- The dataset download reuses the **same OAuth access token** as a Bearer header on the
  signed URL, then unzips and inventories audio files for the assembly line.

In [ ]:
# ============ CELL 4a: Kaggle MCP connection (OAuth 2.0 / token) ============
from urllib.parse import urlparse, parse_qs            # to pull ?code=... out of the pasted redirect URL
from mcp.client.auth import OAuthClientProvider, TokenStorage
from mcp.shared.auth import OAuthClientMetadata
from contextlib import AsyncExitStack
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

KAGGLE_MCP_URL = "https://www.kaggle.com/mcp"          # Kaggle's official MCP endpoint
KAGGLE_TOKEN   = ""                                     # OPTIONAL fallback: paste a KGAT... token from
                                                        # kaggle.com -> Settings -> API -> Generate New Token

# Token cache lives in a KERNEL GLOBAL so re-running this cell does NOT lose the tokens
# you already earned (no repeated consent/paste). Survives cell re-runs; cleared on kernel restart.
KAGGLE_OAUTH_CACHE = globals().get("KAGGLE_OAUTH_CACHE") or {"tokens": None, "client_info": None}

class InMemoryTokenStorage(TokenStorage):              # reads/writes the kernel-global cache above
    async def get_tokens(self):            return KAGGLE_OAUTH_CACHE["tokens"]
    async def set_tokens(self, tokens):    KAGGLE_OAUTH_CACHE["tokens"] = tokens
    async def get_client_info(self):       return KAGGLE_OAUTH_CACHE["client_info"]
    async def set_client_info(self, info): KAGGLE_OAUTH_CACHE["client_info"] = info

async def show_authorization_url(authorization_url):   # headless lab -> we print the consent URL for YOUR browser
    print("1) Open this URL in your browser and approve access:\n\n" + authorization_url)

async def wait_for_pasted_callback():                  # browser lands on dead localhost URL; paste it back here
    pasted_url = input("\n2) Paste the FULL localhost redirect URL here: ").strip()
    query = parse_qs(urlparse(pasted_url).query)       # extract the authorization code (+ state) from it
    return query["code"][0], query.get("state", [None])[0]

token_storage = InMemoryTokenStorage()                 # keep a handle so the download step can reuse the token
kaggle_oauth = OAuthClientProvider(                    # the SDK does registration + code exchange + refresh
    server_url=KAGGLE_MCP_URL,
    client_metadata=OAuthClientMetadata(
        client_name="call-moderator-v4-notebook",
        redirect_uris=["http://localhost:8765/callback"],
        grant_types=["authorization_code", "refresh_token"],
        response_types=["code"],
        token_endpoint_auth_method="none"),            # REQUIRED by Kaggle: public PKCE client
    storage=token_storage,
    redirect_handler=show_authorization_url,
    callback_handler=wait_for_pasted_callback)

if "connection_stack" in globals():                    # re-run safety: close the previous MCP connection
    try:
        await connection_stack.aclose()
    except Exception:
        pass
connection_stack = AsyncExitStack()                    # keeps the MCP session alive across cells
if KAGGLE_TOKEN:                                       # token route: explicit Bearer header, no browser needed
    client_context = streamablehttp_client(KAGGLE_MCP_URL,
                                           headers={"Authorization": f"Bearer {KAGGLE_TOKEN}"})
else:                                                  # OAuth route: SDK challenges only if the server demands it
    client_context = streamablehttp_client(KAGGLE_MCP_URL, auth=kaggle_oauth)
read_stream, write_stream, _ = await connection_stack.enter_async_context(client_context)
kaggle_session = await connection_stack.enter_async_context(ClientSession(read_stream, write_stream))
await kaggle_session.initialize()                      # MCP handshake

# IMPORTANT: Kaggle accepts ANONYMOUS connections without ever showing a login prompt —
# tools list fine but real calls fail with "An error occurred invoking ...". Verify identity:
def raw_payload(call_result):                          # flatten EVERYTHING the server sent back (text or not)
    parts = [getattr(c, "text", None) or repr(c) for c in call_result.content]
    return "\n".join(parts)

KAGGLE_TOOLS = (await kaggle_session.list_tools()).tools           # live tool list = source of truth
print(f"Connected — {len(KAGGLE_TOOLS)} Kaggle tools available")

# Authentication check. NOTE: individual Kaggle tools can fail for their own reasons (missing
# args, server quirks), so a failing probe does NOT always mean "not logged in" — the
# `authorize` tool is the authority: "already authorized" = you are in, proceed to 4b.
identity_probe = await kaggle_session.call_tool("get_user_profile", {})
if not identity_probe.isError:
    print("Authenticated:", raw_payload(identity_probe)[:300])
else:
    try:                                               # triggers the OAuth consent flow on first ever run
        authorize_reply = await kaggle_session.call_tool("authorize", {})
        print("authorize:", raw_payload(authorize_reply))
    except Exception as quirk:                         # Kaggle replies off-spec with a BARE STRING
        if "already authorized" in str(quirk):
            print("AUTHORIZED (Kaggle: 'User is already authorized') -> proceed to CELL 4b.")
            print("(the get_user_profile probe is unreliable — it fails even when logged in)")
        else:
            print(f"authorize quirk: {str(quirk)[:200]}")
            print("-> if a URL was printed above: open, approve, paste — then RE-RUN this cell")

In [ ]:
# ============ CELL 4b: search + download the recordings ============
import httpx, zipfile, pathlib                          # download the signed URL + unzip locally

def result_payload(call_result):                        # MCP result -> python data (JSON-decode when possible)
    text = "\n".join(c.text for c in call_result.content if getattr(c, "text", None))
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

async def call_kaggle(tool_name, argument_variants):    # try several arg spellings — Kaggle's schema can evolve
    last_error = None
    for arguments in argument_variants:
        result = await kaggle_session.call_tool(tool_name, arguments)
        if not result.isError:
            return result_payload(result)
        last_error = result_payload(result)
    raise RuntimeError(f"{tool_name} failed: {last_error}")

def find_tool(*name_keywords):                          # pick a tool by keywords from the live list
    for tool in KAGGLE_TOOLS:
        if all(k in tool.name.lower() for k in name_keywords):
            return tool.name

def input_schema(tool_name):                            # the server's own argument spec = ground truth
    return next((t.inputSchema or {} for t in KAGGLE_TOOLS if t.name == tool_name), {})

def build_args(tool_name, value):                       # schema-aware argument builder
    schema = input_schema(tool_name)
    properties = schema.get("properties", {})
    if list(properties) == ["request"]:                 # Kaggle wraps ALL args in a nested {"request": {...}}
        inner = properties["request"].get("properties", {})
        for key in inner:                               # pick the most likely inner field for our value
            if "search" in key.lower():
                return {"request": {key: value}}
        return {"request": {}}
    for key in (schema.get("required") or list(properties)):
        if properties.get(key, {}).get("type", "string") == "string":
            return {key: value}
    return {}

print("search_datasets schema  :", json.dumps(input_schema("search_datasets"))[:300])
print("download_dataset schema :", json.dumps(input_schema("download_dataset"))[:300], "\n")

try:                                                    # search is DISCOVERY ONLY — never fatal
    search_results = await call_kaggle("search_datasets",   # Kaggle wraps args in {"request": {...}}
        [{"request": {"searchNullable": "call center audio"}},
         build_args("search_datasets", "call center audio")])
    preview = search_results if isinstance(search_results, str) else json.dumps(search_results, indent=2)
    print(preview[:1200])                               # eyeball the results, then set the reference below
except Exception as search_error:
    print(f"search skipped ({str(search_error)[:120]}) — continuing with the known DATASET_REFERENCE")

DATASET_REFERENCE = "unidpro/call-center-audio"         # owner/slug — CHANGE based on the search output
owner_slug, dataset_slug = DATASET_REFERENCE.split("/")
download_info = await call_kaggle("download_dataset",   # Kaggle wraps args in {"request": {...}}
    [{"request": {"ownerSlug": owner_slug, "datasetSlug": dataset_slug}},
     {"request": {"ownerSlug": owner_slug, "datasetSlug": dataset_slug,
                  "datasetVersionNumberNullable": None, "fileNameNullable": None}}])

download_url = re.search(r"https?://[^\s\"']+",         # grab the first URL wherever it sits in the payload
    download_info if isinstance(download_info, str) else json.dumps(download_info)).group(0)

DATA_DIR = pathlib.Path("kaggle_call_data"); DATA_DIR.mkdir(exist_ok=True)
archive_path = DATA_DIR / "dataset_download"
request_headers = {}
stored_tokens = await token_storage.get_tokens()        # reuse the OAuth access token for the file download
if stored_tokens:
    request_headers["Authorization"] = f"Bearer {stored_tokens.access_token}"
with httpx.stream("GET", download_url, headers=request_headers, follow_redirects=True, timeout=300) as response:
    response.raise_for_status()                         # fail loudly on a bad URL
    with open(archive_path, "wb") as output_file:
        for chunk in response.iter_bytes():             # stream to disk — no giant buffer in RAM
            output_file.write(chunk)
if zipfile.is_zipfile(archive_path):                    # Kaggle ships zips; extract in place
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATA_DIR)

AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
audio_files = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in AUDIO_EXTENSIONS)
print(f"{len(audio_files)} audio files ready; testing with up to 3")

## CELL 4c — Assess the dataset (audio QA before trusting transcripts)

Published call datasets are usually **redacted**: PII and profanity are overwritten with censor
bleeps, and Whisper notoriously *hallucinates words over pure tones and silence* — which can
produce false flags downstream. This cell profiles every recording before the pipeline runs:
duration, channels, silence share, and how many 5s chunks look like **pure tones** (one FFT
bin dominating the spectrum = beep, not speech). High beep/silence counts mean transcripts
need the confidence gates that CELL 5 now applies (`no_speech_prob`, `avg_logprob`,
`compression_ratio` filters).

In [ ]:
# ============ CELL 4c: dataset audio QA ============
import numpy as np, librosa

def profile_recording(audio_path, chunk_seconds=5.0, sample_rate=16000):
    samples, _ = librosa.load(audio_path, sr=sample_rate, mono=False)   # keep channel layout
    channels = samples if samples.ndim == 2 else samples[np.newaxis, :]
    chunk_len = int(chunk_seconds * sample_rate)
    silent_chunks = beep_chunks = speech_chunks = 0
    for channel in channels:                            # walk every 5s window on every channel
        for start in range(0, channel.shape[0], chunk_len):
            piece = channel[start:start + chunk_len]
            if piece.size == 0 or float(np.abs(piece).mean()) < 1e-4:
                silent_chunks += 1                      # effectively no signal
                continue
            windowed = piece * np.hanning(len(piece))   # tone test: one FFT bin dominating = beep
            spectrum = np.abs(np.fft.rfft(windowed))
            if spectrum.sum() > 1e-6 and spectrum.max() / spectrum.sum() > 0.30:
                beep_chunks += 1
            else:
                speech_chunks += 1
    duration_s = channels.shape[1] / sample_rate
    return {"file": audio_path.name, "duration_s": round(duration_s, 1),
            "channels": channels.shape[0], "speech_chunks": speech_chunks,
            "beep_chunks": beep_chunks, "silent_chunks": silent_chunks}

print(f"{'file':<40}{'dur(s)':>8}{'ch':>4}{'speech':>8}{'beeps':>7}{'silent':>8}")
print("-" * 78)
for audio_path in audio_files:
    p = profile_recording(audio_path)
    print(f"{p['file']:<40}{p['duration_s']:>8}{p['channels']:>4}"
          f"{p['speech_chunks']:>8}{p['beep_chunks']:>7}{p['silent_chunks']:>8}")
print("\nbeep_chunks > 0 confirms censor bleeps -> CELL 5's confidence gates protect the judge")

## CELL 5 — STEP 1: THE EARS (Faster-Whisper producer)

The producer streams each recording in **5-second chunks** (the document's chunk size),
transcribes each chunk with **Faster-Whisper `small` int8 on CPU** — deliberate: CTranslate2
has no ROCm backend, CPU int8 still beats real time on 5s chunks, and the GPU stays 100%
vLLM's — then puts the transcript on the call's **`asyncio.Queue`** stamped with
`transcribed_at` (the moment the flag-latency clock starts).

Stereo recordings get each channel transcribed separately (telephony puts one party per
channel — free speaker separation). `REALTIME_PACING=True` makes the producer sleep 5s per
chunk to mimic a live stream; leave `False` to benchmark flat-out.

In [ ]:
# ============ CELL 5: THE EARS ============
import numpy as np, librosa                             # audio loading / resampling
from faster_whisper import WhisperModel                 # CTranslate2 whisper — fastest CPU transcription

# ACCURACY UPGRADE: 'small' garbled telephony audio. distil-large-v3 has large-v3 accuracy
# distilled for speed — the best accuracy/latency trade-off that still runs on CPU int8.
WHISPER_MODEL_SIZE = "distil-large-v3"                  # set back to "small" to trade accuracy for speed
ears_model = WhisperModel(WHISPER_MODEL_SIZE, device="cpu", compute_type="int8")

CHUNK_SECONDS     = 5.0                                 # live-simulation ingestion chunk size
SAMPLE_RATE       = 16000                               # whisper's native rate
REALTIME_PACING   = False                               # True = live-stream simulation (5s chunks, 1x pace)
DEBUG_TRANSCRIPTS = True                                # print every transcript / skip reason

# ACCURACY UPGRADE 2: fixed 5s windows CUT WORDS AT THE BOUNDARIES and strip acoustic
# context — a major source of garbled text. For recorded files we now transcribe each
# channel IN FULL and let Whisper's VAD find natural utterance boundaries; the 5s chunk
# path remains only for live simulation, where audio genuinely arrives in pieces.

def looks_like_beep(samples: np.ndarray) -> bool:       # censor bleeps = near-pure tones
    """One FFT bin holding a large share of total energy means a tone, not speech."""
    windowed = samples * np.hanning(len(samples))       # reduce spectral leakage
    spectrum = np.abs(np.fft.rfft(windowed))
    total_energy = spectrum.sum()
    return bool(total_energy > 1e-6 and spectrum.max() / total_energy > 0.30)

def quality_gated(segments):                            # Whisper HALLUCINATES over beeps/silence
    """Keep only segments the model itself is confident about."""
    trustworthy = []
    for segment in segments:
        if segment.no_speech_prob > 0.5:                # model doubts speech was present
            continue
        if segment.avg_logprob < -1.0:                  # very low decoding confidence
            continue
        if segment.compression_ratio > 2.4:             # repetitive output = hallucination marker
            continue
        trustworthy.append(segment)
    return trustworthy

def transcribe_chunk(samples: np.ndarray) -> str:       # LIVE-SIM path: one 5s piece, fast decode
    segments, _ = ears_model.transcribe(samples, language="en", beam_size=1, vad_filter=True)
    return " ".join(s.text.strip() for s in quality_gated(segments)).strip()

def transcribe_channel_fully(samples: np.ndarray) -> list:   # FILE path: full acoustic context
    segments, _ = ears_model.transcribe(
        samples, language="en",
        beam_size=5,                                    # wider beam = better word choices
        vad_filter=True,                                # natural utterance boundaries, no mid-word cuts
        initial_prompt="Customer service phone call between an agent and a customer.")
    return [{"start": float(s.start), "text": s.text.strip()}
            for s in quality_gated(segments) if s.text.strip()]

async def ears_producer(call_id: str, audio_path, transcript_queue: asyncio.Queue):
    """STEP 1: transcribe -> queue.put. Accurate full-file mode, or paced 5s live simulation."""
    samples, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=False)    # keep channels separate
    channels = samples if samples.ndim == 2 else samples[np.newaxis, :] # mono -> fake 1-channel array

    if not REALTIME_PACING:                             # ===== ACCURATE FILE MODE =====
        utterances = []
        for channel_index in range(channels.shape[0]):  # each channel = one (anonymous) speaker
            asr_started = time.perf_counter()
            channel_segments = await asyncio.to_thread(transcribe_channel_fully, channels[channel_index])
            per_segment_asr_ms = ((time.perf_counter() - asr_started) * 1000
                                  / max(len(channel_segments), 1))      # amortized ASR cost per utterance
            for segment in channel_segments:
                utterances.append({"call_id": call_id, "speaker": f"speaker_{channel_index}",
                                   "text": segment["text"], "start": segment["start"],
                                   "chunk_index": int(segment["start"] // CHUNK_SECONDS),
                                   "asr_ms": per_segment_asr_ms})
        for utterance in sorted(utterances, key=lambda u: u["start"]):  # interleave -> conversation order
            utterance["transcribed_at"] = time.perf_counter()           # flag-latency clock starts HERE
            if DEBUG_TRANSCRIPTS:
                print(f"[{call_id} {utterance['speaker']} @{utterance['start']:6.1f}s] "
                      f"\"{utterance['text']}\"", flush=True)
            await transcript_queue.put(utterance)       # ZERO-COPY handshake 1->2
        await transcript_queue.put(None)                # sentinel: this call's audio is finished
        return

    # ===== LIVE-STREAM SIMULATION ===== (5s chunks, paced at 1x)
    chunk_len = int(CHUNK_SECONDS * SAMPLE_RATE)
    total_chunks = int(np.ceil(channels.shape[1] / chunk_len))
    for chunk_index in range(total_chunks):
        for channel_index in range(channels.shape[0]):
            piece = channels[channel_index, chunk_index * chunk_len:(chunk_index + 1) * chunk_len]
            chunk_tag = f"[{call_id} ch{channel_index} chunk{chunk_index:03d}]"
            if piece.size == 0 or float(np.abs(piece).mean()) < 1e-4:
                if DEBUG_TRANSCRIPTS: print(f"{chunk_tag} SKIP silence", flush=True)
                continue
            if looks_like_beep(piece):
                if DEBUG_TRANSCRIPTS: print(f"{chunk_tag} SKIP censor-beep tone", flush=True)
                continue
            asr_started = time.perf_counter()
            text = await asyncio.to_thread(transcribe_chunk, piece)     # off the event loop
            asr_ms = (time.perf_counter() - asr_started) * 1000
            if not text:
                if DEBUG_TRANSCRIPTS: print(f"{chunk_tag} SKIP no confident speech", flush=True)
                continue
            if DEBUG_TRANSCRIPTS: print(f"{chunk_tag} \"{text}\"", flush=True)
            await transcript_queue.put({"call_id": call_id, "speaker": f"speaker_{channel_index}",
                                        "text": text, "chunk_index": chunk_index, "asr_ms": asr_ms,
                                        "transcribed_at": time.perf_counter()})
        await asyncio.sleep(CHUNK_SECONDS)              # pace at 1x — like a live call
    await transcript_queue.put(None)

print(f"THE EARS ready (faster-whisper {WHISPER_MODEL_SIZE} / int8 / CPU, "
      f"{'live-sim 5s chunks' if REALTIME_PACING else 'accurate full-file mode'})")

## CELL 6 — STEP 2: THE BRAIN (queue consumer → vLLM judge → escalation rules)

One Brain worker **per call** pulls from that call's queue with `await queue.get()` —
FIFO order within a call is preserved (rule 3 needs consecutive sentiment), while the 3
calls run in parallel and vLLM batches their requests together.

Roles are guessed by a zero-token marker-phrase scoreboard that updates as the call
progresses (reps say "thank you for calling"; customers say "my bill"). The judge prompt
holds only the last 3 utterances — small context = low latency = fewer prompt tokens.
Escalation stays deterministic code; on a flag the worker pushes to the **alert queue**
(handshake 2→3) and keeps processing.

In [ ]:
# ============ CELL 6: THE BRAIN ============
TURN_ANALYSIS_SCHEMA = {                                # the judge can ONLY answer in this shape
    "type": "object",
    "properties": {
        "sentiment":  {"enum": [-2, -1, 0, 1, 2]},      # customer sentiment, clamped by grammar
        "violations": {"type": "array", "items": {"enum": list(POLICY)}},  # only real codes possible
        "reason":     {"type": "string", "maxLength": 90},                 # short audit note
    },
    "required": ["sentiment", "violations", "reason"],
}

_policy_lines = "\n".join(f"{code}: {definition} [{severity}]"            # compact policy digest for the prompt
                          for code, (severity, definition) in POLICY.items())
MODERATOR_SYSTEM_PROMPT = (                             # ~300 tokens, sent with every judge call
    "You are a strict call-center compliance moderator. Judge ONLY the LATEST utterance, using context.\n"
    "Violation codes:\n" + _policy_lines + "\n"
    "Company policy: " + COMPANY_POLICY_ALLOWANCES + "\n"
    "A venting customer is NOT a violation unless threats/abuse (C2). A rep REFUSING an improper "
    "request is NOT a violation. Report sentiment as the customer's sentiment right now (-2..2).")

REP_MARKER_PHRASES = ("thank you for calling", "how can i help", "how may i help", "this is",
                      "let me check", "i apologize", "is there anything else", "have a great day")
CUSTOMER_MARKER_PHRASES = ("my bill", "my account", "i was charged", "i'm calling", "im calling",
                           "i want a refund", "cancel my", "not working", "speak to a")

def new_call_state():                                   # everything the Brain remembers about one call
    return {"context": deque(maxlen=3),                 # rolling window: last 3 utterances only (token diet)
            "role_scores": defaultdict(int),            # marker scoreboard per anonymous speaker
            "violations_log": [],                       # [(turn_number, role, code)] for the rules + audit
            "sentiment_history": [],                    # customer sentiment trajectory (rule 3)
            "turn_number": 0,                           # running counter
            "escalated": False}                         # flag latch — first trigger wins

def guess_role(state, speaker_tag, text):               # 0-token live role guess, improves as evidence arrives
    lowered = text.lower()
    state["role_scores"][speaker_tag] += (sum(p in lowered for p in REP_MARKER_PHRASES)
                                          - sum(p in lowered for p in CUSTOMER_MARKER_PHRASES))
    scores = state["role_scores"]                       # whoever scores highest is currently 'rep'
    best_rep = max(scores, key=lambda s: scores[s])
    return "rep" if speaker_tag == best_rep and scores[best_rep] > 0 else "customer"

CALL_STATE = {}                                         # call_id -> state (the in-memory 'db' of live calls)

async def brain_worker(call_id: str, transcript_queue: asyncio.Queue, alert_queue: asyncio.Queue):
    """STEP 2: await queue.get() -> judge -> rules -> (maybe) alert. One worker per call."""
    state = CALL_STATE.setdefault(call_id, new_call_state())
    while True:
        item = await transcript_queue.get()             # sleeps here until THE EARS delivers (or sentinel)
        if item is None:                                # sentinel = this call's audio is finished
            break
        dequeued_at = time.perf_counter()               # when the Brain actually picked this turn up
        state["turn_number"] += 1
        turn_number = state["turn_number"]
        role = guess_role(state, item["speaker"], item["text"])          # rep vs customer, free
        hints = keyword_prescreen(item["text"])         # 0-token candidate codes for the judge to verify
        hint_note = f"\nRegex hints to verify (may be false alarms): {hints}" if hints else ""
        context = "\n".join(state["context"]) or "(call start)"          # last 3 utterances max
        analysis = await generate_json("turn_moderation", MODERATOR_SYSTEM_PROMPT,
                                       f'Context:\n{context}\n\nLATEST {role.upper()} utterance: '
                                       f'"{item["text"]}"{hint_note}',
                                       TURN_ANALYSIS_SCHEMA, max_tokens=64)
        judged_at = time.perf_counter()                 # verdict timestamp (flag latency = judged_at - transcribed_at)
        state["context"].append(f"{role}: {item['text']}")               # update rolling window AFTER judging
        violations = [c for c in analysis.get("violations", []) if c in POLICY]
        sentiment = analysis.get("sentiment", 0)
        state["violations_log"] += [(turn_number, role, c) for c in violations]
        if role == "customer":
            state["sentiment_history"].append(sentiment)
        queue_wait_ms = (dequeued_at - item["transcribed_at"]) * 1000   # time spent waiting in the FIFO
        judge_ms      = (judged_at - dequeued_at) * 1000                # pure LLM verdict time
        total_ms      = item.get("asr_ms", 0) + queue_wait_ms + judge_ms  # spoken-audio -> verdict logged
        violation_note = f"   <- {violations}: {analysis.get('reason', '')}" if violations else ""
        print(f"[{call_id} t{turn_number:02d}] {role:>8} ({sentiment:+d}): "
              f"{item['text'][:100]}{violation_note}\n"
              f"        timing: asr {item.get('asr_ms', 0):.0f}ms | queue {queue_wait_ms:.0f}ms | "
              f"judge {judge_ms:.0f}ms | audio->verdict {total_ms:.0f}ms", flush=True)
        audit_db.execute("INSERT INTO turns VALUES (?,?,?,?,?,?,?)",     # in-memory audit trail, every turn
                         (call_id, turn_number, role, item["text"],
                          sentiment, ",".join(violations), analysis.get("reason", "")))
        if not state["escalated"]:                      # the three deterministic rules, in priority order
            severities = [SEVERITY_BY_CODE[c] for _, _, c in state["violations_log"]]
            recent = state["sentiment_history"][-2:]
            rule = None
            if "critical" in severities:
                rule = f"rule 1: critical violation {violations}"
            elif severities.count("high") >= 2:
                rule = "rule 2: two high-severity violations"
            elif len(recent) == 2 and all(s <= -2 for s in recent):
                rule = "rule 3: sustained very negative customer sentiment"
            if rule:
                state["escalated"] = True               # latch — one alarm per call
                await alert_queue.put({                 # ZERO-COPY handshake 2->3
                    "call_id": call_id, "turn_number": turn_number, "rule": rule,
                    "detail": f'{role}: "{item["text"][:120]}"',
                    "transcribed_at": item["transcribed_at"],            # end-to-end clock start
                    "dequeued_at": dequeued_at,                           # separates queue wait from judge time
                    "flagged_at": judged_at})

print("THE BRAIN ready (1 worker per call, last-3-utterance context, deterministic rules)")

## CELL 7 — STEP 3: THE ALARM (instant flag + in-memory audit DB)

Replaces the document's PyQt6 red-flash window with what Jupyter can do **instantly**: a
flushed, unmissable alert line the moment the Brain pushes to the alert queue, plus a row in
a **sqlite `:memory:`** database (the in-memory DB) so every flag and every judged turn is
queryable afterwards. The printed latency is the headline metric: *transcript ready → flag
raised*, in milliseconds.

In [ ]:
# ============ CELL 7: THE ALARM ============
audit_db = sqlite3.connect(":memory:")                  # in-memory DB: zero I/O latency, queryable with SQL
audit_db.execute("CREATE TABLE turns  (call_id TEXT, turn_number INT, role TEXT, text TEXT, "
                 "sentiment INT, violations TEXT, reason TEXT)")        # every judged utterance
audit_db.execute("CREATE TABLE alerts (call_id TEXT, turn_number INT, rule TEXT, detail TEXT, "
                 "flag_latency_ms REAL)")                               # every raised flag

FLAG_LATENCIES_MS = []                                  # collected for the final stats cell

async def alarm_consumer(alert_queue: asyncio.Queue):
    """STEP 3: await alert -> print INSTANTLY (flush=True) + record to the audit DB."""
    while True:
        alert = await alert_queue.get()                 # sleeps until the Brain raises a flag (or sentinel)
        if alert is None:                               # sentinel = pipeline shutting down
            break
        flag_latency_ms  = (alert["flagged_at"] - alert["transcribed_at"]) * 1000  # end-to-end
        queue_wait_ms    = (alert.get("dequeued_at", alert["transcribed_at"]) - alert["transcribed_at"]) * 1000
        judge_ms         = (alert["flagged_at"] - alert.get("dequeued_at", alert["transcribed_at"])) * 1000
        FLAG_LATENCIES_MS.append(judge_ms)              # the honest real-time number: pure judge latency
        audit_db.execute("INSERT INTO alerts VALUES (?,?,?,?,?)",
                         (alert["call_id"], alert["turn_number"], alert["rule"],
                          alert["detail"], flag_latency_ms))
        print("\n" + "!" * 78, flush=True)              # flush=True -> appears the instant it happens
        print(f"!! ESCALATE  {alert['call_id']}  turn {alert['turn_number']}  ({alert['rule']})", flush=True)
        print(f"!! {alert['detail']}", flush=True)
        print(f"!! judge latency: {judge_ms:.0f} ms  (+{queue_wait_ms:.0f} ms queue wait in "
              f"flat-out benchmark mode; end-to-end {flag_latency_ms:.0f} ms)", flush=True)
        # NOTE: queue wait exists because REALTIME_PACING=False floods all chunks at once.
        # With live pacing (1 chunk / 5s) the queue stays empty whenever judge < 5s.
        print("!" * 78 + "\n", flush=True)

print("THE ALARM ready (instant flush + sqlite :memory: audit)")

## CELL 8 — Run the assembly line

Wires everything: one transcript queue + one Ears producer + one Brain worker **per call**,
one shared alert queue + one Alarm consumer, all under a single `asyncio.gather`. With
`REALTIME_PACING=False` this is a flat-out benchmark; set it `True` in CELL 5 to watch flags
fire against a simulated live stream.

In [ ]:
# ============ CELL 8: orchestration ============
MAX_CALLS = 3                                           # the 1-3 test recordings requested
selected_recordings = audio_files[:MAX_CALLS]           # take the first N from the Kaggle download
assert selected_recordings, "No audio files found — check CELL 4b output / DATASET_REFERENCE"

alert_queue = asyncio.Queue()                           # shared Brain -> Alarm channel
alarm_task  = asyncio.create_task(alarm_consumer(alert_queue))         # start STEP 3 first (always listening)

pipeline_started = time.perf_counter()                  # total wall-clock timer
per_call_tasks = []
for audio_path in selected_recordings:                  # build one independent lane per call
    call_id = audio_path.stem                           # filename (no extension) as the call's id
    transcript_queue = asyncio.Queue()                  # the zero-copy handshake 1->2 for THIS call
    per_call_tasks.append(ears_producer(call_id, audio_path, transcript_queue))   # STEP 1 (produces + sentinel)
    per_call_tasks.append(brain_worker(call_id, transcript_queue, alert_queue))   # STEP 2 (consumes until sentinel)

await asyncio.gather(*per_call_tasks)                   # run all lanes concurrently; returns when audio is done
await alert_queue.put(None)                             # sentinel: tell the Alarm to finish
await alarm_task                                        # wait for the last alert to be printed/stored
pipeline_seconds = time.perf_counter() - pipeline_started

total_turns = audit_db.execute("SELECT COUNT(*) FROM turns").fetchone()[0]
print(f"assembly line done: {len(selected_recordings)} call(s), {total_turns} judged utterances "
      f"in {pipeline_seconds:.1f}s ({total_turns/max(pipeline_seconds,0.001):.1f} utterances/s)")

## CELL 9 — Results: flags, latency, tokens

Three readouts, all from in-memory data: the **alerts** (what was flagged, which rule, how
fast), **latency** percentiles per stage (judge latency *is* the flag latency), and exact
**token usage** per stage with a combined total.

In [ ]:
# ============ CELL 9: results ============
print("RAISED FLAGS")                                   # straight from the in-memory audit DB
for row in audit_db.execute("SELECT call_id, turn_number, rule, flag_latency_ms FROM alerts"):
    print(f"  {row[0]:<28} turn {row[1]:<3} {row[2]:<48} {row[3]:>7.0f} ms")
if not FLAG_LATENCIES_MS:
    print("  (no escalations triggered on these recordings)")

def percentile(values, fraction):                       # tiny helper — avoids a numpy dependency here
    ordered = sorted(values)
    return ordered[min(int(len(ordered) * fraction), len(ordered) - 1)]

print("\nLATENCY (ms)")                                 # judge latency == transcript->flag latency per turn
for stage, samples in STAGE_LATENCY_MS.items():
    print(f"  {stage:<22} n={len(samples):<5} avg={sum(samples)/len(samples):>7.0f}  "
          f"p95={percentile(samples, 0.95):>7.0f}  max={max(samples):>7.0f}")
if FLAG_LATENCIES_MS:
    print(f"  {'transcript->flag':<22} n={len(FLAG_LATENCIES_MS):<5} "
          f"avg={sum(FLAG_LATENCIES_MS)/len(FLAG_LATENCIES_MS):>7.0f}  "
          f"max={max(FLAG_LATENCIES_MS):>7.0f}")

print("\nTOKENS BY STAGE")                              # exact counts from vLLM's usage field
grand = {"calls": 0, "prompt": 0, "completion": 0}
for stage, usage in sorted(STAGE_TOKEN_USAGE.items()):
    total = usage["prompt"] + usage["completion"]
    print(f"  {stage:<22} calls={usage['calls']:<5} prompt={usage['prompt']:>8,} "
          f"completion={usage['completion']:>8,} total={total:>9,}")
    for field in grand:
        grand[field] += usage[field]
print(f"  {'COMBINED':<22} calls={grand['calls']:<5} prompt={grand['prompt']:>8,} "
      f"completion={grand['completion']:>8,} total={grand['prompt']+grand['completion']:>9,}")

print("\nPER-CALL SUMMARY")                             # SQL over the in-memory audit trail
for row in audit_db.execute(
    "SELECT call_id, COUNT(*), SUM(violations != ''), MIN(sentiment) FROM turns GROUP BY call_id"):
    print(f"  {row[0]:<28} turns={row[1]:<4} turns_with_violations={row[2] or 0:<3} "
          f"worst_sentiment={row[3]}")

## Design notes

- **Why `asyncio.Queue` and not LangChain/LangGraph:** the pipeline is a fixed 3-stage line —
  the architecture document itself specifies in-process queues. A graph framework would add
  abstraction layers (and milliseconds) between transcript and flag for zero functional gain.
- **Why one queue/worker per call:** FIFO order within a call is required (rule 3 reads
  *consecutive* sentiment), while calls stay parallel and vLLM batches across them. The
  shared `Semaphore(16)` is the global concurrency gate.
- **Where the speed comes from:** silence chunks are dropped before transcription (0 cost),
  the prompt carries only the last 3 utterances, guided JSON kills retries, and the alarm
  consumer prints with `flush=True` the instant the Brain pushes — flag latency is one judge
  call (~hundreds of ms on a 4B model), measured and reported per flag.
- **To go truly live:** replace `ears_producer`'s file loop with a WebSocket/WebRTC receiver
  pushing 5s chunks — every line downstream of `transcript_queue.put(...)` stays identical.
- **Real-time rehearsal:** set `REALTIME_PACING = True` in CELL 5 to pace chunks at 1×.

## CELL 10 — Full transcripts (from the audit DB)

Everything the judge actually saw, per call and in order — the ground truth for debugging
false flags: compare these lines against the audio to spot ASR errors, and against the
alert log to see exactly which words triggered which code.

In [ ]:
# ============ CELL 10: transcript dump ============
for (call_id,) in audit_db.execute("SELECT DISTINCT call_id FROM turns"):
    print("=" * 78); print(f"TRANSCRIPT {call_id}"); print("=" * 78)
    rows = audit_db.execute(
        "SELECT turn_number, role, text, sentiment, violations, reason "
        "FROM turns WHERE call_id = ? ORDER BY turn_number", (call_id,))
    for turn_number, role, text, sentiment, violations, reason in rows:
        flag_note = f"   <- {violations}: {reason}" if violations else ""
        print(f"[t{turn_number:02d}] {role:>8} ({sentiment:+d}): {text}{flag_note}")
    print()